# PanTS Dataset
## Exploratory Data Analysis
### Things to note:
- I dont possess the Test set. It is reserved for third-party evaluation by the developing team.
- The dataset is divided into chunks. chunk 1 → 00000001–00001000, chunk 2 → 00001001–00002000, ...
  - I have only downloaded the first chunk for now.
- `metadata.xlsx` does not contain the full labels. The complete metadata is only accessible via the test set.
- `metadata.xlsx` TR has the following labels: (PanTS ID, shape, spacing, ct phase,	sex,	age,	manufacturer,	manufacturer model,	study type,	site,	site detail,	site nationality,	study year,	tumor?,	structured report)
- `ImageTr` contains the actual CT scans
  - uses `.nii.gz` file format
  - 0001 - 1000 files
- `LabelTr` contains the annotations for those CT scans (with expert-validated annotations)
  - uses `.nii.gz` file format
  - 0001 - 9901 files
  - it contains the segmentation masks for the corresponding CT scans

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from config import LABEL_DIR, IMAGE_DIR

import os
import numpy as np
import nibabel
import monai
from monai.transforms import LoadImage

In [4]:
loader = LoadImage(image_only=True)

case_id = "PanTS_00000001"
segmentations = f"{LABEL_DIR}/{case_id}/segmentations"

print("Segmentation masks ---->")
for f in sorted(os.listdir(segmentations)):
    print(f)

num_masks = sorted(os.listdir(segmentations))
print(f"Number of masks: {len(num_masks)}")

Segmentation masks ---->
adrenal_gland_left.nii.gz
adrenal_gland_right.nii.gz
aorta.nii.gz
bladder.nii.gz
celiac_artery.nii.gz
colon.nii.gz
common_bile_duct.nii.gz
duodenum.nii.gz
femur_left.nii.gz
femur_right.nii.gz
gall_bladder.nii.gz
kidney_left.nii.gz
kidney_right.nii.gz
liver.nii.gz
lung_left.nii.gz
lung_right.nii.gz
pancreas.nii.gz
pancreas_body.nii.gz
pancreas_head.nii.gz
pancreas_tail.nii.gz
pancreatic_duct.nii.gz
pancreatic_lesion.nii.gz
postcava.nii.gz
prostate.nii.gz
spleen.nii.gz
stomach.nii.gz
superior_mesenteric_artery.nii.gz
veins.nii.gz
Number of masks: 28


NOTE:
- The dataset only has combined venous segmentation mask per case. This limitation will need to be acknowledged in the thesis. 

In [5]:
combined_labels = loader(f"{LABEL_DIR}/{case_id}/combined_labels.nii.gz")

print("Combined labels ---->")
print(f"Shape: {tuple(combined_labels.shape)}")
print(f"Voxel dimensions: {combined_labels.meta['pixdim'][1:4]}")
print(f"Unique label values: {np.unique(combined_labels.numpy())}")

Combined labels ---->
Shape: (512, 333, 200)
Voxel dimensions: [0.625 0.625 0.8  ]
Unique label values: [ 0.  1.  2.  3.  5.  6.  7.  8. 11. 12. 13. 14. 15. 16. 17. 18. 19. 20.
 21. 22. 24. 25. 26. 27.]


# Notes:
- `Voxel dimensions: [0.625 0.625 0.8  ]` -> spacing is different in z
- TODO: might need to resample all volumnes to isotropic spacing before training `Voxel dimensions: [0.625 0.625 0.8  ]`

In [6]:
lesion = loader(f"{segmentations}/pancreatic_lesion.nii.gz")

print(f"Lesion shape: {tuple(lesion.shape)}")
print(f"Lesion unique values: {np.unique(lesion)}")
print(f"Voxels: {np.count_nonzero(lesion)}")

Lesion shape: (512, 333, 200)
Lesion unique values: [0.]
Voxels: 0


In [8]:
vascular_masks = [
    "superior_mesenteric_artery.nii.gz",
    "celiac_artery.nii.gz",
    "veins.nii.gz",
    "postcava.nii.gz",
    "aorta.nii.gz"
]

for mask_name in vascular_masks:
    path = f"{segmentations}/{mask_name}"
    img = loader(path)
    data = img.numpy()
    print(f"\n=== {mask_name} ===")
    print(f"Shape: {data.shape}")
    print(f"Unique vals: {np.unique(data)}")
    print(f"Voxels: {np.count_nonzero(data)}")


=== superior_mesenteric_artery.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 3349

=== celiac_artery.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 1356

=== veins.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 13537

=== postcava.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 52615

=== aorta.nii.gz ===
Shape: (512, 333, 200)
Unique vals: [0. 1.]
Voxels: 156992


# Notes:
- All 5 masks are the same shape.
- All masks are clean binary with only 0 and 1 and there is no partial values, no noise.
- TODO: Use these masks to compute NCCN based circumferential contact angle

In [7]:
cases = sorted(os.listdir(LABEL_DIR))
lesion_counts = {}

for case in cases:
    lesion_path = f"{LABEL_DIR}/{case}/segmentations/pancreatic_lesion.nii.gz"
    if os.path.exists(lesion_path):
        data = loader(lesion_path).numpy()
        lesion_counts[case] = np.count_nonzero(data)

positive_cases = {k: v for k, v in lesion_counts.items() if v > 0}
print(f"Total cases: {len(cases)}")
print(f"Tumour-positive: {len(positive_cases)}")
print(f"Tumour-negative: {len(cases) - len(positive_cases)}")

Total cases: 9902
Tumour-positive: 1033
Tumour-negative: 8869


In [ ]:
# Among positive cases, which have vascular mask overlap with lesion?
vascular_masks = [
    "superior_mesenteric_artery.nii.gz",
    "celiac_artery.nii.gz",
    "veins.nii.gz",
    "postcava.nii.gz",
    "aorta.nii.gz"
]

contact_cases = []

for case in positive_cases:
    seg_path = f"{LABEL_DIR}/{case}/segmentations"
    lesion_bool = loader(f"{seg_path}/pancreatic_lesion.nii.gz").numpy() > 0

    for vessel in vascular_masks:
        vessel_path = f"{seg_path}/{vessel}"
        if os.path.exists(vessel_path):
            vessel_data = loader(vessel_path).numpy() > 0
            if np.any(lesion_bool & vessel_data):
                contact_cases.append({"case": case, "vessel": vessel})
                break

print(f"Positive cases with tumour–vessel mask overlap: {len(contact_cases)}")
print(f"Positive cases without tumour–vessel mask overlap: {len(positive_cases) - len(contact_cases)}")

Positive cases with tumour–vessel mask overlap: 186
Positive cases without tumour–vessel mask overlap: 847


# Notes:
- 847 (~8.6%) -> Tumour-positive + no direct tumour–vessel mask overlap
- 186 (~1.9%) -> Tumour-positive + direct tumour–vessel mask overlap
- 8868 (~89.6%) -> Tumour-negative

- Should it be a binary resectability framing?
  - **Class 0**: Resectable
  - **Class 1**: Non-resectable

Labels will be derived from tumour–vessel geometric measurements following NCCN-inspired criteria


In [18]:
# Get first tumour-positive case
positive_case = list(positive_cases.keys())[0]
seg_path = f"{LABEL_DIR}/{positive_case}/segmentations"

print(f"Case: {positive_case} [POSITIVE]")

# Lesion mask
lesion = loader(f"{seg_path}/pancreatic_lesion.nii.gz").numpy()
print(f"\n=== Lesion ===")
print(f"Shape:         {tuple(lesion.shape)}")
print(f"Unique vals:   {np.unique(lesion)}")
print(f"Non-zero:      {np.count_nonzero(lesion)}")
print(f"Tumour present: {np.count_nonzero(lesion) > 0}")

# Vascular masks
vascular_masks = [
    "superior_mesenteric_artery.nii.gz",
    "celiac_artery.nii.gz",
    "veins.nii.gz",
    "postcava.nii.gz",
    "aorta.nii.gz"
]

lesion_bool = lesion > 0

for mask_name in vascular_masks:
    vessel = loader(f"{seg_path}/{mask_name}").numpy()
    vessel_bool = vessel > 0
    overlap = np.any(lesion_bool & vessel_bool)
    print(f"\n=== {mask_name} ===")
    print(f"Non-zero:       {np.count_nonzero(vessel)}")
    print(f"Vessel present: {np.count_nonzero(vessel) > 0}")
    print(f"Lesion contact: {overlap}")

Case: PanTS_00000003 [POSITIVE]

=== Lesion ===
Shape:         (495, 349, 40)
Unique vals:   [0. 1.]
Non-zero:      1055
Tumour present: True

=== superior_mesenteric_artery.nii.gz ===
Non-zero:       487
Vessel present: True
Lesion contact: False

=== celiac_artery.nii.gz ===
Non-zero:       843
Vessel present: True
Lesion contact: False

=== veins.nii.gz ===
Non-zero:       4866
Vessel present: True
Lesion contact: False

=== postcava.nii.gz ===
Non-zero:       16278
Vessel present: True
Lesion contact: False

=== aorta.nii.gz ===
Non-zero:       22067
Vessel present: True
Lesion contact: False


# Notes:
- Shape is different compared to PanTS_00000001.
- TODO: will require resampling to a consistent size and spacing before training

In [19]:
ct = loader(f"{IMAGE_DIR}/{positive_case}/ct.nii.gz").numpy()
print(f"CT shape: {tuple(ct.shape)}")

CT shape: (495, 349, 40)


# Notes:
- CT and masks are same in shape

In [20]:
print(contact_cases[:3])

[{'case': 'PanTS_00000086', 'vessel': 'veins.nii.gz'}, {'case': 'PanTS_00000167', 'vessel': 'veins.nii.gz'}, {'case': 'PanTS_00000231', 'vessel': 'veins.nii.gz'}]


In [21]:
case = "PanTS_00000086"
seg_path = f"{LABEL_DIR}/{case}/segmentations"

print(f"Case: {case} [POSITIVE]")

# Lesion mask
lesion = loader(f"{seg_path}/pancreatic_lesion.nii.gz").numpy()
print(f"\n=== Lesion ===")
print(f"Shape:          {tuple(lesion.shape)}")
print(f"Unique vals:    {np.unique(lesion)}")
print(f"Non-zero:       {np.count_nonzero(lesion)}")
print(f"Tumour present: {np.count_nonzero(lesion) > 0}")

# Vascular masks
lesion_bool = lesion > 0

for mask_name in vascular_masks:
    vessel = loader(f"{seg_path}/{mask_name}").numpy()
    vessel_bool = vessel > 0
    overlap = np.any(lesion_bool & vessel_bool)
    print(f"\n=== {mask_name} ===")
    print(f"Non-zero:       {np.count_nonzero(vessel)}")
    print(f"Vessel present: {np.count_nonzero(vessel) > 0}")
    print(f"Lesion contact: {overlap}")

Case: PanTS_00000086 [POSITIVE]

=== Lesion ===
Shape:          (455, 371, 75)
Unique vals:    [0. 1.]
Non-zero:       6556
Tumour present: True

=== superior_mesenteric_artery.nii.gz ===
Non-zero:       2355
Vessel present: True
Lesion contact: False

=== celiac_artery.nii.gz ===
Non-zero:       1637
Vessel present: True
Lesion contact: False

=== veins.nii.gz ===
Non-zero:       27273
Vessel present: True
Lesion contact: True

=== postcava.nii.gz ===
Non-zero:       61551
Vessel present: True
Lesion contact: False

=== aorta.nii.gz ===
Non-zero:       65521
Vessel present: True
Lesion contact: False


In [22]:
# check CT scans shape
shapes = []
for case in cases:
    ct_path = f"{IMAGE_DIR}/{case}/ct.nii.gz"
    if os.path.exists(ct_path):
        img = loader(ct_path)
        shapes.append(tuple(img.shape))

shapes = np.array(shapes)
print(f"Min shape: {shapes.min(axis=0)}")
print(f"Max shape: {shapes.max(axis=0)}")
print(f"Mean shape: {shapes.mean(axis=0).astype(int)}")

Min shape: [43 57  8]
Max shape: [ 753  512 1060]
Mean shape: [437 343 183]


# NOTES:
- Z-axis (slices) is most extreme --> 8 to 1,060 (132x difference)
- Reflects multi-site acquisition
- TODO: Resample to fixed target shape and isotropic spacing

In [10]:
from scipy.ndimage import distance_transform_edt

sma = loader(
    f"{seg_path}/superior_mesenteric_artery.nii.gz"
).numpy() > 0

distance_map = distance_transform_edt(~sma)

tumour_distances = distance_map[lesion]

print("Closest tumour-SMA distance:",
      tumour_distances.min(),
      "voxels")

Closest tumour-SMA distance: 20.518284528683193 voxels


In [11]:
case = "PanTS_00000086"

seg_path = f"{LABEL_DIR}/{case}/segmentations"

lesion = loader(
    f"{seg_path}/pancreatic_lesion.nii.gz"
).numpy() > 0

for vessel in vascular_masks:
    vessel_path = f"{seg_path}/{vessel}"

    if os.path.exists(vessel_path):
        vessel_mask = loader(vessel_path).numpy() > 0

        overlap = np.sum(lesion & vessel_mask)

        print(vessel, "overlap voxels:", overlap)

superior_mesenteric_artery.nii.gz overlap voxels: 0
celiac_artery.nii.gz overlap voxels: 0
veins.nii.gz overlap voxels: 150
postcava.nii.gz overlap voxels: 0
aorta.nii.gz overlap voxels: 0


In [12]:
from scipy.ndimage import distance_transform_edt

case = "PanTS_00000086"
seg_path = f"{LABEL_DIR}/{case}/segmentations"

tumour = loader(
    f"{seg_path}/pancreatic_lesion.nii.gz"
).numpy() > 0


for vessel_name in [
    "superior_mesenteric_artery.nii.gz",
    "celiac_artery.nii.gz",
    "veins.nii.gz"
]:

    vessel = loader(
        f"{seg_path}/{vessel_name}"
    ).numpy() > 0

    distance_map = distance_transform_edt(~vessel)

    closest_distance = distance_map[tumour].min()

    print(
        vessel_name,
        "closest distance:",
        closest_distance,
        "voxels"
    )

superior_mesenteric_artery.nii.gz closest distance: 16.1245154965971 voxels
celiac_artery.nii.gz closest distance: 9.433981132056603 voxels
veins.nii.gz closest distance: 0.0 voxels


# Positive cases check
The veins segmentation masks from the PanTS dataset are not one single vessel. Instead, it is a tree-like structure. This makes it hard to compute the centreline because which part to keep to then run the angle computation through? One positive case was tested and keeping the dominant venous strcuture and using lcc filtering can work. 
This check is to check among tumour-positive cases, how many have direct voxel overlap with each vascular segmentation mask?

In [8]:
positive_case_ids = list(positive_cases.keys())

print(len(positive_case_ids))
print(positive_case_ids[:10])

1033
['PanTS_00000003', 'PanTS_00000026', 'PanTS_00000029', 'PanTS_00000031', 'PanTS_00000035', 'PanTS_00000036', 'PanTS_00000040', 'PanTS_00000042', 'PanTS_00000044', 'PanTS_00000047']


In [11]:
contact_case_ids = [x["case"] for x in contact_cases]

print(len(contact_case_ids))
print(contact_case_ids[:20])

186
['PanTS_00000086', 'PanTS_00000167', 'PanTS_00000231', 'PanTS_00000246', 'PanTS_00000270', 'PanTS_00000363', 'PanTS_00000449', 'PanTS_00000465', 'PanTS_00000554', 'PanTS_00000605', 'PanTS_00000693', 'PanTS_00000696', 'PanTS_00000911', 'PanTS_00001077', 'PanTS_00001183', 'PanTS_00001326', 'PanTS_00001347', 'PanTS_00001351', 'PanTS_00001436', 'PanTS_00001536']


## Check which vessels are most commonly involved
#### Among tumour-positive cases with direct tumour–vessel voxel overlap, which vessels are involved most frequently?

In [17]:
from collections import Counter
import pandas as pd

vessel_counts = Counter(
    x["vessel"] for x in contact_cases
)

vessel_counts

df = (
    pd.DataFrame(
        vessel_counts.items(),
        columns=["Vessel", "Cases"]
    )
    .sort_values("Cases", ascending=False)
    .reset_index(drop=True)
)

df

,Vessel,Cases
0,veins.nii.gz,108
1,celiac_artery.nii.gz,48
2,postcava.nii.gz,21
3,superior_mesenteric_artery.nii.gz,8
4,aorta.nii.gz,1


In [13]:
case = contact_case_ids[3]

seg_path = f"{LABEL_DIR}/{case}/segmentations"

print(case)

for f in os.listdir(seg_path):
    if f.endswith(".nii.gz"):
        print(f)

PanTS_00000246
adrenal_gland_left.nii.gz
adrenal_gland_right.nii.gz
aorta.nii.gz
bladder.nii.gz
celiac_artery.nii.gz
colon.nii.gz
common_bile_duct.nii.gz
duodenum.nii.gz
femur_left.nii.gz
femur_right.nii.gz
gall_bladder.nii.gz
kidney_left.nii.gz
kidney_right.nii.gz
liver.nii.gz
lung_left.nii.gz
lung_right.nii.gz
pancreas.nii.gz
pancreas_body.nii.gz
pancreas_head.nii.gz
pancreas_tail.nii.gz
pancreatic_duct.nii.gz
pancreatic_lesion.nii.gz
postcava.nii.gz
prostate.nii.gz
spleen.nii.gz
stomach.nii.gz
superior_mesenteric_artery.nii.gz
veins.nii.gz


In [14]:
lesion = loader(
    f"{seg_path}/pancreatic_lesion.nii.gz"
).numpy() > 0

veins = loader(
    f"{seg_path}/veins.nii.gz"
).numpy() > 0

sma = loader(
    f"{seg_path}/superior_mesenteric_artery.nii.gz"
).numpy() > 0

In [15]:
print("Tumor voxels:", lesion.sum())
print("Vein voxels:", veins.sum())
print("SMA voxels:", sma.sum())

print(
    "Tumor-vein overlap:",
    np.logical_and(lesion, veins).sum()
)

print(
    "Tumor-SMA overlap:",
    np.logical_and(lesion, sma).sum()
)

Tumor voxels: 264178
Vein voxels: 18117
SMA voxels: 1448
Tumor-vein overlap: 0
Tumor-SMA overlap: 0


In [16]:
import pandas as pd

rows = []

for item in contact_cases:
    case = item["case"]
    vessel = item["vessel"]

    seg_path = f"{LABEL_DIR}/{case}/segmentations"

    lesion = loader(
        f"{seg_path}/pancreatic_lesion.nii.gz"
    ).numpy() > 0

    vessel_mask = loader(
        f"{seg_path}/{vessel}"
    ).numpy() > 0

    overlap = np.logical_and(
        lesion,
        vessel_mask
    ).sum()

    rows.append({
        "case": case,
        "vessel": vessel,
        "tumor_voxels": lesion.sum(),
        "vessel_voxels": vessel_mask.sum(),
        "overlap_voxels": overlap
    })


df_contact = pd.DataFrame(rows)

df_contact.head()

,case,vessel,tumor_voxels,vessel_voxels,overlap_voxels
0,PanTS_00000086,veins.nii.gz,6556,27273,150
1,PanTS_00000167,veins.nii.gz,1296,27484,47
2,PanTS_00000231,veins.nii.gz,1256,29535,31
3,PanTS_00000246,celiac_artery.nii.gz,264178,6021,2
4,PanTS_00000270,celiac_artery.nii.gz,23437,3429,410
